# GRIT on Modal

Run this notebook locally in Jupyter or VS Code from the GRIT repository. It controls remote GPUs through `modal_app.py`; the local machine does not need CUDA or vLLM.

Flow: **connect to Modal → prepare/reuse artifacts → start the fresh 40-step run → inspect checkpoints**.

Default target: pinned Qwen2.5-3B-Instruct, 5 responses/prompt, microbatch 1, projected AdamW delta and central difference. `A100-80GB:3` is split into **2 actors + 1 vLLM**. Actors use FP32. Data/checkpoints live on Modal Volumes.

The workflow intentionally skips smoke/resume jobs and starts the requested full run. GPU cells are paid operations; run them only when ready.


## 1. Install the Modal client and sign in

Run the next cell once. Then sign in from a terminal in the **same Python environment** with `python -m modal setup`. For gated models or datasets, create a Modal Secret containing `HF_TOKEN` and set `HF_SECRET` below; never put the token in this notebook.


In [ ]:
print('[grit-02] Starting...', flush=True)
import os
import subprocess
import sys
def run_with_progress(command, *, env=None):
    # Relay child stdout/stderr through the notebook's Python output stream.
    # text=True converts tqdm carriage returns into readable progress lines.
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env={**os.environ, **(env or {}), 'PYTHONUNBUFFERED': '1'}) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

run_with_progress([sys.executable, "-m", "pip", "install", "modal>=1,<2", "pandas>=2,<3", "matplotlib>=3.8,<4"])

print('[grit-02] Complete.', flush=True)


## 2. Configure the server and run

The last GPU serves vLLM and the other two are replicated actors. This trainer does not shard model or optimizer state. Set a Modal Secret containing `HF_TOKEN` and a destination Hub repository before starting.

The two replicated actors offload FP32 state into one shared container. The Modal function requests 256 GiB of host memory; Modal must accept and schedule that request before this configuration is considered runnable. Two actors process more examples each, so the 40-step run may exceed Modal's 24-hour Function timeout.


In [ ]:
print('[grit-04] Starting...', flush=True)
from pathlib import Path
from datetime import datetime, timezone
import importlib
import json
import os

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modal_app.py").is_file()), None)
assert REPO is not None, "Open main.ipynb from the GRIT repository."
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

GPU = "A100-80GB:3"
DATA_VOLUME = "grit-data"
OUTPUT_VOLUME = "grit-checkpoints"
HF_SECRET = ""  # Required: Modal Secret containing HF_TOKEN.
HUB_REPO_ID = ""  # Required destination, for example username/grit-qwen25-3b.
os.environ.update(GRIT_MODAL_GPU=GPU, GRIT_DATA_VOLUME=DATA_VOLUME, GRIT_OUTPUT_VOLUME=OUTPUT_VOLUME)
if HF_SECRET:
    os.environ["GRIT_HF_SECRET"] = HF_SECRET
else:
    os.environ.pop("GRIT_HF_SECRET", None)

import modal
import modal_app
modal_app = importlib.reload(modal_app)
app = modal_app.app
RUN_ID = datetime.now(timezone.utc).strftime("grit-%Y%m%d-%H%M%S")
MODEL = "Qwen/Qwen2.5-3B-Instruct"
print({"repo": str(REPO), "gpu": GPU, "run": RUN_ID})

print('[grit-04] Complete.', flush=True)


## 3. Prepare or reuse data on the Volume

This cell creates task prompts, 1,000 projector contexts (334 general, 333 math, 333 code), and a separate set of 1,920 KL trust-region contexts (640 per domain, 48 × 40 steps), with no shared prompts. Both use the same frozen base revision and tokenizer; artifacts use model-specific split_v1 paths. Complete artifacts are reused, while incomplete output is retained for inspection. The first preparation downloads the model and datasets and generates contexts. It does not run again after each optimizer step.

The trainer also requires `/data/eval/evaluation_prompts.parquet`, supplied independently with a unique nonempty `prompt` column. Preparation does not derive evaluation prompts from training sources. The full run stops before model training if this file is missing or overlaps either preservation set.


In [ ]:
print('[grit-06] Starting...', flush=True)
with modal.enable_output(), app.run():
    bundle = modal_app.prepare.remote(MODEL)
print(json.dumps(bundle, indent=2))

print('[grit-06] Complete.', flush=True)


## 4. Configure training

`projector_relaxation=0.05` uses the relaxed projector `Q=P+0.05(I-P)`: it passes 5% of the direction outside the null space, while the trust region controls predictor drift. Use `0` for the hard NSPO projector or `1` to disable projection. `task_optimizer="sgd"` selects the gradient predictor; `"adamw"` selects the AdamW delta and its matching correction. `use_curvature=False` selects the first-order variant. `lambda_pres=0` selects projection only.

Sampling uses temperature=1 and top-p=1. GRPO uses the sampled-token log-probabilities from vLLM as the denominator and does not recompute advantages during central differences. Actor weights and forwards are FP32 with TF32 disabled; `rollout_dtype` applies only to vLLM. The stored base top-64 plus tail anchor is generated with the preparation dtype, and step 1 measures its KL against the FP32 actor before the predictor.

The projector and KL sets come from train or unlabeled sources after normalized-prompt deduplication. The KL set contributes to model updates, so it is not evaluation data. Preparation selects 1,920 balanced KL contexts from the 6,000-prompt pool, exactly 48 samples for each of 40 steps.


In [ ]:
print('[grit-08] Starting...', flush=True)
config = {
    **bundle,
    "output_dir": f"/checkpoints/{RUN_ID}",
    "task_optimizer": "adamw",
    "projector_relaxation": 0.05,
    "task_batch_size": 321,  # Global prompts; split 161/160 over two actors.
    "preserve_batch_size": 48,  # Global KL contexts; 24 per actor.
    "generations": 5,
    "max_prompt_length": 512,
    "max_response_length": 256,
    "max_preserve_length": 2304,
    "lr": 1e-6,
    "lambda_pres": 1.0,
    "epsilon_pres": 0.05,
    "top_k": 64,  # KL support: frozen-base top-64 plus one tail class.
    "base_statistics_dtype": "float16",  # Base preparation forward dtype.
    "use_curvature": True,
    "fd_radius": 0.05,
    "fd_check_interval": 10,  # Compare HVP(rho) with HVP(2*rho); 0 disables the diagnostic pass.
    "gradient_topk": 5,       # Store norms only, never gradient tensors.
    "rollout_dtype": "auto",
    "rollout_memory_utilization": 0.8,
    "gradient_checkpointing": True,
    "progress": True,
    "max_steps": 40, # 34 complete steps use 10,914 prompts; step 35 starts the reshuffled epoch.
    "save_steps": 2,
    "hub_repo_id": HUB_REPO_ID,
    "seed": 66,
}
print(json.dumps(config, indent=2))

print('[grit-08] Complete.', flush=True)


## 7. Start the full run

This run starts from the base model in a new output directory. The detached call continues if the notebook disconnects. Save the call ID so you can reconnect. Modal prints the dashboard and log URL at launch.


In [ ]:
print('[grit-14] Starting...', flush=True)
assert HF_SECRET, "Set HF_SECRET to a Modal Secret containing HF_TOKEN before launch."
assert HUB_REPO_ID, "Set HUB_REPO_ID before launch."
with modal.enable_output(), app.run(detach=True):
    job = modal_app.train.spawn(config)
    CALL_ID = job.object_id
    print("Modal call ID:", CALL_ID)

record = {"call_id": CALL_ID, "config": config, "output_volume": OUTPUT_VOLUME}
Path("outputs").mkdir(exist_ok=True)
Path("outputs/modal_run.json").write_text(json.dumps(record, indent=2))

print('[grit-14] Complete.', flush=True)


## 8. Reconnect and collect the result

This cell waits for the job to finish. After a kernel restart, read `outputs/modal_run.json` to recover the call ID. Checkpoints remain on the Volume after the app stops.


In [ ]:
print('[grit-16] Starting...', flush=True)
record = json.loads(Path("outputs/modal_run.json").read_text())
job = modal.FunctionCall.from_id(record["call_id"])
result = job.get()
print(json.dumps(result, indent=2))

print('[grit-16] Complete.', flush=True)


## 9. Inspect metrics, diagnostics, or a checkpoint

The command below is printed only; it does not download all weights automatically. The path inside the Volume omits the `/checkpoints/` prefix.


In [ ]:
print('[grit-18] Starting...', flush=True)
with modal.enable_output(), app.run():
    status = modal_app.inspect_run.remote(record["config"]["output_dir"])
print(json.dumps(status, indent=2))
if status["checkpoints"]:
    remote_path = str(Path(status["checkpoints"][-1]).relative_to("/checkpoints"))
    print(f"python -m modal volume get {record['output_volume']} {remote_path} ./checkpoints/modal-download")

print('[grit-18] Complete.', flush=True)


## 10. Post-run diagnostic plots

`alerts` contains per-step warnings, while `diagnostics` summarizes the run. The plots below expose reward collapse, blocked task updates, dominant corrections, trust projection errors, timing bottlenecks, vLLM/actor mismatch, and central-difference sensitivity to the radius.


In [ ]:
print('[grit-20] Starting...', flush=True)
import pandas as pd
import matplotlib.pyplot as plt

metrics = pd.DataFrame(status["metrics"]).set_index("step")
display(pd.Series(status["diagnostics"], name="run diagnostics").to_frame())
alerts = metrics[metrics["alerts"].map(bool)][["alerts"]]
display(alerts if len(alerts) else "No automatic-threshold alerts.")
fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True)
metrics[["reward_mean", "reward_std", "unsafe_fraction"]].plot(ax=axes[0, 0], title="Reward / safety")
metrics[["task_grad_norm", "preservation_grad_norm", "hvp_norm"]].plot(ax=axes[0, 1], logy=True, title="Gradient norms")
metrics[["hard_projection_retained_fraction", "projection_retained_fraction", "correction_to_task_ratio", "relative_update_norm"]].plot(ax=axes[1, 0], logy=True, title="Hard vs relaxed update")
metrics[["trust_kl_mean", "trust_kl_max", "trust_projected_kl_max"]].plot(ax=axes[1, 1], logy=True, title="Trust region KL")
axes[1, 1].axhline(record["config"]["epsilon_pres"], color="black", linestyle="--", label="epsilon")
metrics[["rollout_seconds", "reward_seconds", "task_backward_seconds", "preservation_backward_seconds", "central_difference_seconds", "checkpoint_seconds"]].plot(ax=axes[2, 0], title="Step time")
metrics[["rollout_actor_logprob_mae", "fd_relative_error_rho_vs_2rho", "fd_one_sided_relative_error"]].plot(ax=axes[2, 1], logy=True, title="Numerical checks")
plt.tight_layout()
plt.show()

print('[grit-20] Complete.', flush=True)


This notebook deliberately contains no smoke or resume launch. A fresh output directory is required for the full run.

Documentation: [Modal in Jupyter](https://modal.com/docs/guide/jupyter-notebooks), [GPU](https://modal.com/docs/guide/gpu), [Volumes](https://modal.com/docs/guide/volumes), [vLLM LLM API](https://docs.vllm.ai/en/v0.15.1/api/vllm/entrypoints/llm/).
